# Ladybug nightly + Icebug: Star Wars graph analytics with Arrow CSR

This notebook uses the `ladybug==0.17.0.dev20260522` nightly build to demonstrate `create_arrow_rel_table` with `layout="CSR"`. Both Ladybug and Icebug consume the same Arrow CSR arrays directly — no intermediate native storage needed. The Arrow CSR arrays are generated using icebug_format.convert_arrow_tables_to_csr

The dataset is intentionally small and friendly: a Star Wars interaction graph with characters, factions, homeworlds, and weighted relationships.


## Environment

Dependencies are declared in `pyproject.toml`. From a fresh checkout, start Jupyter with:

```bash
uv run jupyter notebook ladybug_icebug_starwars.ipynb
```

Or execute the notebook from the CLI with:

```bash
uv run jupyter nbconvert --to notebook --execute ladybug_icebug_starwars.ipynb --inplace
```

In [23]:
# Optional setup cell: sync dependencies from pyproject.toml.
# If you launched with `uv run jupyter notebook`, this is already done.
!uv sync

Resolved 105 packages in 1ms
Audited 100 packages in 1ms


In [1]:
import ladybug
import icebug as ib
import pyarrow as pa

print("ladybug", ladybug.__version__)
print("icebug", ib.__version__)


ladybug 0.17.0.20260522
icebug 12.7


## Build a friendly Arrow graph

Ladybug can register Arrow tables directly as graph node and relationship tables. Relationship tables use endpoint columns named `from` and `to`, with values pointing at node row ids.

In [2]:
characters = pa.table({
    "id": pa.array(list(range(18)), type=pa.uint64()),
    "name": [
        "Luke Skywalker", "Leia Organa", "Han Solo", "Chewbacca", "R2-D2", "C-3PO",
        "Obi-Wan Kenobi", "Yoda", "Darth Vader", "Emperor Palpatine", "Tarkin",
        "Boba Fett", "Lando Calrissian", "Jabba the Hutt", "Mon Mothma",
        "Wedge Antilles", "Mara Jade", "Ahsoka Tano",
    ],
    "faction": [
        "Rebellion", "Rebellion", "Rebellion", "Rebellion", "Rebellion", "Rebellion",
        "Jedi", "Jedi", "Empire", "Empire", "Empire", "Underworld", "Rebellion",
        "Underworld", "Rebellion", "Rebellion", "Underworld", "Jedi",
    ],
    "homeworld": [
        "Tatooine", "Alderaan", "Corellia", "Kashyyyk", "Naboo", "Tatooine",
        "Stewjon", "Dagobah", "Tatooine", "Naboo", "Eriadu", "Kamino", "Socorro",
        "Nal Hutta", "Chandrila", "Corellia", "Coruscant", "Shili",
    ],
    "role": [
        "Jedi hopeful", "General", "Smuggler", "Co-pilot", "Astromech", "Protocol droid",
        "Mentor", "Grand master", "Sith lord", "Emperor", "Grand Moff", "Bounty hunter",
        "Administrator", "Crime lord", "Chancellor", "Pilot", "Agent", "Fulcrum",
    ],
})

name_to_id = dict(zip(characters["name"].to_pylist(), characters["id"].to_pylist()))

interaction_names = pa.table({
    "source": [
        "Luke Skywalker", "Luke Skywalker", "Han Solo", "Leia Organa", "R2-D2", "R2-D2",
        "C-3PO", "Obi-Wan Kenobi", "Yoda", "Obi-Wan Kenobi", "Ahsoka Tano", "Ahsoka Tano",
        "Darth Vader", "Darth Vader", "Tarkin", "Darth Vader", "Darth Vader", "Darth Vader",
        "Boba Fett", "Boba Fett", "Jabba the Hutt", "Jabba the Hutt", "Lando Calrissian",
        "Lando Calrissian", "Mon Mothma", "Mon Mothma", "Wedge Antilles", "Wedge Antilles",
        "Mara Jade", "Mara Jade", "Mara Jade",
    ],
    "target": [
        "Leia Organa", "Han Solo", "Chewbacca", "Han Solo", "C-3PO", "Luke Skywalker",
        "Leia Organa", "Luke Skywalker", "Luke Skywalker", "Yoda", "Obi-Wan Kenobi", "Yoda",
        "Emperor Palpatine", "Tarkin", "Emperor Palpatine", "Luke Skywalker", "Leia Organa",
        "Obi-Wan Kenobi", "Darth Vader", "Jabba the Hutt", "Han Solo", "Leia Organa",
        "Han Solo", "Leia Organa", "Leia Organa", "Wedge Antilles", "Luke Skywalker",
        "Lando Calrissian", "Emperor Palpatine", "Luke Skywalker", "Boba Fett",
    ],
    "scene": [
        "twins and rebellion", "rescues and raids", "lifelong crew", "command and chemistry",
        "droid duo", "secret plans", "translation and diplomacy", "training", "training",
        "jedi council", "clone wars", "jedi order", "master and apprentice", "imperial command",
        "death star politics", "family conflict", "imperial pursuit", "old master", "bounty contract",
        "underworld contract", "debt", "palace rescue", "old friends", "alliance planning",
        "rebel leadership", "fleet command", "rogue squadron", "battle of endor", "emperor's hand",
        "rivalry to trust", "underworld overlap",
    ],
    "strength": pa.array([8, 7, 10, 8, 9, 7, 5, 9, 9, 6, 6, 5, 10, 7, 6, 10, 6, 9, 6, 8, 8, 4, 8, 6, 8, 5, 7, 6, 7, 6, 4], type=pa.int64()),
})

interactions = pa.table({
    "from": pa.array([name_to_id[name] for name in interaction_names["source"].to_pylist()], type=pa.uint64()),
    "to": pa.array([name_to_id[name] for name in interaction_names["target"].to_pylist()], type=pa.uint64()),
    "scene": interaction_names["scene"],
    "strength": interaction_names["strength"],
})

characters, interactions

(pyarrow.Table
 id: uint64
 name: string
 faction: string
 homeworld: string
 role: string
 ----
 id: [[0,1,2,3,4,...,13,14,15,16,17]]
 name: [["Luke Skywalker","Leia Organa","Han Solo","Chewbacca","R2-D2",...,"Jabba the Hutt","Mon Mothma","Wedge Antilles","Mara Jade","Ahsoka Tano"]]
 faction: [["Rebellion","Rebellion","Rebellion","Rebellion","Rebellion",...,"Underworld","Rebellion","Rebellion","Underworld","Jedi"]]
 homeworld: [["Tatooine","Alderaan","Corellia","Kashyyyk","Naboo",...,"Nal Hutta","Chandrila","Corellia","Coruscant","Shili"]]
 role: [["Jedi hopeful","General","Smuggler","Co-pilot","Astromech",...,"Crime lord","Chancellor","Pilot","Agent","Fulcrum"]],
 pyarrow.Table
 from: uint64
 to: uint64
 scene: string
 strength: int64
 ----
 from: [[0,0,2,1,4,...,15,15,16,16,16]]
 to: [[1,2,3,2,5,...,0,12,9,0,11]]
 scene: [["twins and rebellion","rescues and raids","lifelong crew","command and chemistry","droid duo",...,"rogue squadron","battle of endor","emperor's hand","rivalry to 

## Build Arrow CSR graph

Since `ladybug==0.17.0.dev20260522`, `create_arrow_rel_table` accepts `layout="CSR"` with a companion `indptr_table`. Once we have the converted the tables into csr format using `icebug_format`, the same Arrow arrays are handed to both Ladybug and Icebug — neither needs to go through native storage.


In [18]:
from icebug_format import IcebugMemGraph

# CSR layout requires edges sorted by source node.
sorted_interactions = interactions.sort_by("from")

# build csr strutures using icebug_format.convert_arrow_tables_to_csr
icemem_graph = IcebugMemGraph.from_arrow_tables(characters, sorted_interactions, undirected=True);

print(sorted_interactions.num_rows)
print(icemem_graph.src.num_rows)
print(icemem_graph.dest.num_rows)
print(icemem_graph.indices.num_rows)
print(icemem_graph.indptr.num_rows)

31
18
18
62
19


In [7]:
undirected_db = ladybug.Database()
undirected = ladybug.Connection(undirected_db)
undirected.create_arrow_table("Character", characters)
undirected.create_arrow_rel_table(
    "INTERACTS_WITH", icemem_graph.indices, "Character", "Character",
    "CSR", icemem_graph.indptr
)

RuntimeError: Runtime exception: Arrow CSR relationship indices table must include destination column 'to'

## Ladybug and Icebug both read the Arrow CSR table directly

**Ladybug** runs Cypher against the CSR-layout Arrow relationship table as-is.

**Icebug** reads the same CSR Arrow arrays directly into `Graph.fromCSR` — no `.csr()` extraction call needed. Also using `fromIcebugMemGraph`.


In [7]:
# Ladybug: degree centrality via Cypher on the Arrow CSR table
undirected.query_as_arrow("""
MATCH (a:Character)-[:INTERACTS_WITH]->(b:Character)
RETURN a.name AS character, a.faction AS faction,
       count(DISTINCT b) AS degree
ORDER BY degree DESC, character
""", 4096).get_as_arrow().to_pandas()


RuntimeError: Binder exception: Table INTERACTS_WITH does not exist.

In [19]:
graph = ib.Graph.fromCSR(icemem_graph.src.num_rows, False, icemem_graph.indices.column("target"), icemem_graph.indptr.column(0))

leiden = ib.community.ParallelLeidenView(graph, iterations=4, gamma=1.0, randomize=False)
leiden.run()
partition = leiden.getPartition()
modularity = ib.community.Modularity().getQuality(partition, graph)

characters_df = undirected.query_as_arrow(
    "MATCH (c:Character) RETURN c.rowid AS rowid, c.id AS id, c.name AS name, c.faction AS faction ORDER BY c.rowid",
    4096,
).get_as_arrow().to_pandas()

communities = characters_df.copy()
communities["community"] = [partition.subsetOf(i) for i in range(graph.numberOfNodes())]
communities = communities.sort_values(["community", "faction", "name"]).reset_index(drop=True)

print(f"Icebug graph: {graph.numberOfNodes()} nodes, {graph.numberOfEdges()} undirected edges")
print(f"Leiden found {partition.numberOfSubsets()} communities; modularity Q={modularity:.4f}")
communities

Icebug graph: 18 nodes, 31 undirected edges
Leiden found 4 communities; modularity Q=0.3730


,rowid,id,name,faction,community
0,17,17,Ahsoka Tano,Jedi,0
1,6,6,Obi-Wan Kenobi,Jedi,0
2,7,7,Yoda,Jedi,0
3,0,0,Luke Skywalker,Rebellion,0
4,3,3,Chewbacca,Rebellion,1
5,2,2,Han Solo,Rebellion,1
6,12,12,Lando Calrissian,Rebellion,1
7,1,1,Leia Organa,Rebellion,1
8,14,14,Mon Mothma,Rebellion,1
9,15,15,Wedge Antilles,Rebellion,1


In [20]:
communities.groupby(["community", "faction"]).size().rename("characters").reset_index()\
    .sort_values(["community", "characters"], ascending=[True, False])

,community,faction,characters
0,0,Jedi,3
1,0,Rebellion,1
2,1,Rebellion,6
3,1,Underworld,1
4,2,Rebellion,2
5,3,Empire,3
6,3,Underworld,2


In [21]:
graph = ib.Graph.fromIcebugMemGraph(icemem_graph, False)

leiden = ib.community.ParallelLeidenView(graph, iterations=4, gamma=1.0, randomize=False)
leiden.run()
partition = leiden.getPartition()
modularity = ib.community.Modularity().getQuality(partition, graph)

characters_df = undirected.query_as_arrow(
    "MATCH (c:Character) RETURN c.rowid AS rowid, c.id AS id, c.name AS name, c.faction AS faction ORDER BY c.rowid",
    4096,
).get_as_arrow().to_pandas()

communities = characters_df.copy()
communities["community"] = [partition.subsetOf(i) for i in range(graph.numberOfNodes())]
communities = communities.sort_values(["community", "faction", "name"]).reset_index(drop=True)

print(f"Icebug graph: {graph.numberOfNodes()} nodes, {graph.numberOfEdges()} undirected edges")
print(f"Leiden found {partition.numberOfSubsets()} communities; modularity Q={modularity:.4f}")
communities

Icebug graph: 18 nodes, 31 undirected edges
Leiden found 4 communities; modularity Q=0.3730


,rowid,id,name,faction,community
0,17,17,Ahsoka Tano,Jedi,0
1,6,6,Obi-Wan Kenobi,Jedi,0
2,7,7,Yoda,Jedi,0
3,0,0,Luke Skywalker,Rebellion,0
4,3,3,Chewbacca,Rebellion,1
5,2,2,Han Solo,Rebellion,1
6,12,12,Lando Calrissian,Rebellion,1
7,1,1,Leia Organa,Rebellion,1
8,14,14,Mon Mothma,Rebellion,1
9,15,15,Wedge Antilles,Rebellion,1


In [22]:
communities.groupby(["community", "faction"]).size().rename("characters").reset_index()\
    .sort_values(["community", "characters"], ascending=[True, False])


,community,faction,characters
0,0,Jedi,3
1,0,Rebellion,1
2,1,Rebellion,6
3,1,Underworld,1
4,2,Rebellion,2
5,3,Empire,3
6,3,Underworld,2


## What happened

1. The Star Wars graph started as ordinary PyArrow tables.
2. IcebugMemGraph is created using `icebug_format.IcebugMemGraph.from_arrow_tables`
4. `create_arrow_rel_table(..., layout="CSR", indptr_table=indptr_table)` registered the Arrow CSR data in Ladybug — no native storage or disk I/O needed.
5. **Ladybug** ran Cypher directly against the CSR-layout Arrow table.
6. **Icebug** consumed the same CSR Arrow arrays directly with `Graph.fromCSR`, `Graph.fromIcebugMemGraph` and ran Leiden community detection.
